# Create a RayCluster with the CodeFlare SDK

This notebook demonstrates how to create and manage a Ray cluster on RHOAI using the CodeFlare SDK.

**Prerequisites:**
- Running in an RHOAI Workbench
- `ray-demo` namespace exists with Kueue configured
- KubeRay operator is running

In [ ]:
from codeflare_sdk import Cluster, ClusterConfiguration

## Define the cluster configuration

In [ ]:
cluster = Cluster(
    ClusterConfiguration(
        name="demo-cluster",
        namespace="ray-demo",
        num_workers=2,
        worker_cpu_requests="100m",
        worker_memory_requests=2,
        image="quay.io/modh/ray:2.47.1-py311-cu121",
        local_queue="default",
    )
)

## RHOAI 3.4.1: AuthenticationReady Workaround



### STOP: Admin Action Required

On RHOAI 3.4.1, the admin must run the auth workaround before proceeding:

```bash
./scripts/fix-auth.sh ray-demo demo-cluster
```

Only proceed to the next cell after the admin confirms the fix is applied.

In [ ]:
> **Important:** On RHOAI 3.4.1, after running `cluster.apply()` below, a cluster administrator must run the auth workaround from a terminal before `cluster.wait_ready()` will succeed:
>
> ```bash
> ./scripts/fix-auth.sh ray-demo demo-cluster
> ```
>
> See [Module 7 - Troubleshooting](https://rrbanda.github.io/rhoai-kuberay/docs/07-troubleshooting) for the full root cause.

## Start the cluster

In [ ]:
cluster.apply()
cluster.wait_ready()

## Check cluster details

In [ ]:
cluster.details()

## Connect with mTLS and run a task

In [ ]:
from codeflare_sdk import generate_cert

generate_cert.generate_tls_cert(cluster.config.name, cluster.config.namespace)
generate_cert.export_env(cluster.config.name, cluster.config.namespace)

In [ ]:
import ray

ray.init(cluster.cluster_uri())
print("Cluster resources:", ray.cluster_resources())
print("Nodes:", len(ray.nodes()))

In [ ]:
@ray.remote
def hello(x):
    import socket
    return f"Task {x} on {socket.gethostname()}"

results = ray.get([hello.remote(i) for i in range(8)])
for r in results:
    print(r)

## Clean up

In [ ]:
ray.shutdown()
cluster.down()